**원본 데이터를 DB에 저장하는 코드.**

#01\. 기본설치

## DB

In [1]:
# [DB 연결 1] postgresql 연결 라이브러리 설치
%pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 36.2 MB/s eta 0:00:00


In [2]:
# [DB 연결 2] 라이브러리
import getpass
import threading
import time
import io               # save_df의 COPY에서 사용
import psycopg2         # PostgreSQL 접속

In [3]:
host   = "YOUR_NGROK_HOST"    # ← ngrok에 표시된 호스트 (tcp:// 제외)
port   = 0                   # ← ngrok에 표시된 포트
dbname = "YOUR_DB_NAME"
user   = "YOUR_DB_USER"
pw     = getpass.getpass("비밀번호: ")



def _connect():
    return psycopg2.connect(
        host=host, port=port, dbname=dbname, user=user, password=pw,
        connect_timeout=15,
        keepalives=1, keepalives_idle=30,          # 연결 유지용 TCP keepalive
        keepalives_interval=10, keepalives_count=5,
    )

conn = _connect()


def ensure_conn():
    """연결이 끊겼으면 재접속 (청크 사이에 호출)"""
    global conn
    try:
        if conn.closed:
            raise psycopg2.InterfaceError("closed")
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
        conn.rollback()                             # 확인용 트랜잭션 정리
    except Exception:
        try:
            conn.close()
        except Exception:
            pass
        conn = _connect()
        print("[재접속 완료]")

with conn.cursor() as cur:
    cur.execute("SELECT current_user, version()")
    print("접속 성공:", cur.fetchone())
conn.rollback()

비밀번호: ··········
접속 성공: ('YOUR_DB_USER', 'PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit')


In [ ]:
# 접속한 사용자와 DB 버전 확인 (PostgreSQL)
with conn.cursor() as cur:
    cur.execute("SELECT current_user, current_database(), version()")
    connected_user, db_name, db_version = cur.fetchone()
conn.rollback()

print("PostgreSQL 연결 성공")
print("접속 사용자:", connected_user)
print("접속 DB:", db_name)
print("버전:", db_version)

PostgreSQL 연결 성공
접속 사용자: YOUR_DB_USER
접속 DB: YOUR_DB_NAME
버전: PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


#IMPORT

In [ ]:
# Colab에서 Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# ============================================================
# [IMPORT] 이 노트북에서 실제로 쓰는 것만 남김
#   (DB 연결 셀의 getpass / oracledb / threading 은 해당 셀에서 별도로 import)
# ============================================================
# 표준 라이브러리
import os                                # 폴더/경로 탐색
import re                                # save_df의 컬럼명 검사, 컬럼명 규칙 검사
import time                              # API 호출 간격, 소요시간 측정
import codecs                            # CSV 인코딩 판별
import zipfile                           # ZIP 내부 파일 읽기
from contextlib import contextmanager    # 파일 열기 헬퍼


# 데이터 처리 / API 호출
import pandas as pd
import requests

# Colab / 노트북 출력
from google.colab import drive, files    # 드라이브 마운트, 직접 업로드
from IPython.display import display

In [ ]:
print(os.listdir("/content/drive/MyDrive/Colab Notebooks/final_project/data"))

['bike_raw', 'subway_raw', 'subway_card_raw', 'subway_processed']


##인증키, 기간 설정

In [ ]:
# 서울 열린데이터광장에서 발급받은 인증키
SEOUL_API_KEY = ""

# 공공데이터포털 API에서 발급받은 인증키_ 역별 승하차인원
DATA_GO_KR_API_KEY = ""

# 전체 수집에 사용할 기간
START_DATE = "2021-01-01"
END_DATE = "2026-09-30"

# 상세 대여이력·고장신고의 첫 응답을 확인할 날짜
# 전체 수집 기간을 변경하는 값은 아닙니다.
CHECK_DATE = "2026-09-30"
CHECK_HOUR = 8

BASE_URL = "http://openapi.seoul.go.kr:8088"

print("대상: 서울 전체 대여소")
print("전체 수집 기간:", START_DATE, "~", END_DATE)
print("응답 확인 날짜:", CHECK_DATE)

대상: 서울 전체 대여소
전체 수집 기간: 2021-01-01 ~ 2026-09-30
응답 확인 날짜: 2026-09-30


#02\. 데이터 수집

In [7]:
# ============================================================
# [DB 저장 공통함수] DataFrame → PostgreSQL 테이블 저장 (COPY 방식)
#   - 사용법은 기존과 동일: save_df(df, "테이블명", conn[, mode="INSERT"])
#   - commit=False 로 호출하면 커밋하지 않음 (호출한 쪽에서 로그와 함께 커밋)
# ============================================================
def save_df(df, table_name, conn, mode=None, commit=True):
    # 허용 옵션 확인: 기존 테이블에 추가할 때만 "INSERT" 지정
    if mode not in (None, "INSERT"):
        raise ValueError('mode는 생략하거나 "INSERT"로 지정하세요.')

    table_name = table_name.lower()                 # PostgreSQL은 소문자로 저장
    columns = [str(c).lower() for c in df.columns]

    # 테이블명·컬럼명 검사
    for name in [table_name, *columns]:
        if not re.fullmatch(r"[a-z][a-z0-9_]{0,62}", name):
            raise ValueError(f"사용할 수 없는 이름: {name}")
    if len(columns) != len(set(columns)):
        raise ValueError("중복된 컬럼명이 있습니다.")
    if df.empty:
        raise ValueError("저장할 데이터가 없습니다.")

    # 원본 보관용: 전부 TEXT로 저장 (앞자리 0 유지). 빈 값은 NULL로 저장됨
    column_sql = ", ".join(f"{c} TEXT" for c in columns)
    col_list = ", ".join(columns)

    # DataFrame을 CSV 텍스트로 만들어 COPY로 한 번에 적재
    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)

    with conn.cursor() as cur:
        try:
            cur.execute("SELECT to_regclass(%s::text)", [table_name])
            table_exists = cur.fetchone()[0] is not None

            if table_exists:
                if mode != "INSERT":
                    raise ValueError(f"{table_name}: 이미 존재하는 테이블입니다.")
                # 기존 테이블에 없는 컬럼이 있으면 중단
                cur.execute("""SELECT column_name FROM information_schema.columns
                               WHERE table_schema = current_schema() AND table_name = %s""",
                            [table_name])
                existing = {r[0] for r in cur.fetchall()}
                missing = [c for c in columns if c not in existing]
                if missing:
                    raise ValueError(f"{table_name}에 없는 컬럼: {missing}")
            else:
                cur.execute(f"CREATE TABLE {table_name} ({column_sql})")

            cur.copy_expert(f"COPY {table_name} ({col_list}) FROM STDIN WITH (FORMAT csv)", buf)
            if commit:
                conn.commit()
        except Exception:
            conn.rollback()      # PostgreSQL은 CREATE도 롤백되므로 별도 정리 불필요
            raise

    print(f"{table_name.upper()}: {len(df):,}행 저장 완료")

##02-1. 따릉이



### api 정상접속 확인

In [8]:
# 이 함수는 첫 5행을 조회합니다.
# 전체 데이터를 받는 페이지 반복 코드는 다음 수집 단계에서 추가합니다.

def check_api(service, date_parameter=""):
    if SEOUL_API_KEY in ("", "본인_인증키"):
        raise ValueError("상단 SEOUL_API_KEY에 본인 인증키를 입력하세요.")

    url = (
        f"{BASE_URL}/{SEOUL_API_KEY}/json/"
        f"{service}/1/5/"
    )

    if date_parameter:
        url += date_parameter

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        data = response.json()

    except (requests.RequestException, ValueError):
        # 인증키가 포함된 URL이 오류 출력에 노출되지 않도록 처리
        raise RuntimeError(
            f"{service}: 연결 또는 JSON 응답 오류입니다."
        ) from None

    # 서비스마다 다른 응답 이름을 찾아 처리
    result_data = next(
        (
            value
            for value in data.values()
            if isinstance(value, dict)
            and "row" in value
        ),
        None
    )

    result = (
        result_data.get("RESULT", {})
        if result_data is not None
        else data.get("RESULT", {})
    )

    code = result.get("CODE")
    message = result.get("MESSAGE")

    if code == "INFO-200":
        print(f"[{service}] 조회된 자료 없음: {date_parameter}")
        return pd.DataFrame()

    if code != "INFO-000" or result_data is None:
        raise RuntimeError(
            f"{service}: {code} / {message}"
        )

    df = pd.DataFrame(result_data["row"])

    print(f"\n서비스: {service}")
    print("응답 전체 건수:", result_data.get("list_total_count"))
    print("이번에 받은 행 수:", len(df))
    print("원본 컬럼:", df.columns.tolist())

    display(df)

    return df

In [ ]:
# 1. 대여소 정보
# 번호·ID·이름·주소·좌표·거치대 수 확인

df_station_check = check_api(
    service="tbCycleStationInfo"
)

# 2. 대여소 마스터 정보
# 대여소ID·주소·좌표 확인
# 이후 대여소 정보와 중복·불일치를 비교

df_station_master_check = check_api(
    service="bikeStationMaster"
)


서비스: tbCycleStationInfo
응답 전체 건수: 3244
이번에 받은 행 수: 5
원본 컬럼: ['STA_LOC', 'RENT_ID', 'RENT_NO', 'RENT_NM', 'RENT_ID_NM', 'HOLD_NUM', 'STA_ADD1', 'STA_ADD2', 'STA_LAT', 'STA_LONG', 'START_INDEX', 'END_INDEX', 'RNUM']


,STA_LOC,RENT_ID,RENT_NO,RENT_NM,RENT_ID_NM,HOLD_NUM,STA_ADD1,STA_ADD2,STA_LAT,STA_LONG,START_INDEX,END_INDEX,RNUM
0,마포구,ST-10,00108,서교동 사거리,108. 서교동 사거리,12,서울특별시 마포구 양화로 93,427,37.55274582,126.91861725,0,0,1
1,광진구,ST-100,00503,더샵스타시티 C동 앞,503. 더샵스타시티 C동 앞,15,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.53666687,127.07359314,0,0,2
2,양천구,ST-1000,00729,서부식자재마트 건너편,729. 서부식자재마트 건너편,10,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51037979,126.86679840,0,0,3
3,양천구,ST-1002,00731,서울시 도로환경관리센터,731. 서울시 도로환경관리센터,10,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.52989960,126.87654114,0,0,4
4,양천구,ST-1003,00732,신월중학교,732. 신월중학교,10,서울특별시 양천구 화곡로 59,신월동 이마트,37.53955078,126.82830048,0,0,5



서비스: bikeStationMaster
응답 전체 건수: 3432
이번에 받은 행 수: 5
원본 컬럼: ['RNTLS_ID', 'ADDR1', 'ADDR2', 'LAT', 'LOT']


,RNTLS_ID,ADDR1,ADDR2,LAT,LOT
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.510380,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.000000,0.000000
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.529900,126.876541


### 전체 데이터

In [9]:
def fetch_all_rows(service, date_parameter=""):
    # sample 키는 전체 수집에 사용할 수 없음
    if SEOUL_API_KEY.strip() in ("", "본인_인증키", "sample"):
        raise ValueError("전체 조회에는 본인 서울 API 인증키가 필요합니다.")

    page_size = 1000
    start = 1
    frames = []
    previous_rows = None

    while True:
        end = start + page_size - 1

        url = f"{BASE_URL}/{SEOUL_API_KEY}/json/{service}/{start}/{end}/"
        if date_parameter:
            url += date_parameter

        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            data = response.json()
        except (requests.RequestException, ValueError):
            raise RuntimeError(f"{service}: {start}~{end}행 요청 실패") from None

        # --- 응답 해석: 정상 / 데이터 없음(INFO-200) / 오류 응답의 구조가 달라 모두 처리 ---
        svc = data.get(service)
        if not isinstance(svc, dict):                      # 서비스 이름이 다르면 첫 딕셔너리 사용
            svc = next((v for v in data.values() if isinstance(v, dict)), {})

        result  = svc.get("RESULT") or data.get("RESULT") or {}   # 서비스 안쪽 → 바깥쪽 순서로 탐색
        code    = result.get("CODE") or data.get("CODE")
        message = result.get("MESSAGE") or data.get("MESSAGE")
        rows    = svc.get("row", [])
        total   = svc.get("list_total_count")

        if code == "INFO-200":                             # 조회된 자료 없음 → 정상 종료
            break

        if code not in (None, "INFO-000"):                 # 인증키 오류 등 → 중단
            raise RuntimeError(f"{service}: {code} / {message}")

        if not rows:
            if code is None:                               # 코드도 행도 없는 알 수 없는 응답
                raise RuntimeError(f"{service}: 알 수 없는 응답 {str(data)[:200]}")
            break

        # 페이지 조건이 무시되어 같은 응답이 반복되는 경우 확인
        if rows == previous_rows:
            raise RuntimeError(f"{service}: 이전 페이지와 동일한 응답입니다.")

        frames.append(pd.DataFrame(rows))
        print(f"{service}: {start}~{end} 요청 → {len(rows):,}행 수신")
        previous_rows = rows

        # 마지막 페이지: 1,000행 미만이거나, 응답의 전체 건수에 도달
        if len(rows) < page_size:
            break
        try:
            if start + len(rows) - 1 >= int(total):
                break
        except (TypeError, ValueError):
            pass

        start += page_size
        time.sleep(0.3)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

####02-1-1. 대여소 정보

1.   대여소정보(반기) > 출처: 서울 자전거 따릉이 (https://www.bikeseoul.com/main.do)
2.   대여소 마스터(수시) >  출처: 서울시 교통정보시스템(https://topis.seoul.go.kr/)

위와 같이 정류소의 자료가 다를 수 있어 2개의 원본 테이블을 생성.

In [ ]:
# 원본 DataFrame을 각각 유지
raw_station = fetch_all_rows("tbCycleStationInfo")
raw_master_station = fetch_all_rows("bikeStationMaster")

print("대여소 정보:", raw_station.shape)
print("대여소 마스터:", raw_master_station.shape)

display(raw_station.head())
display(raw_master_station.head())

# [대여소 DB 저장 ] 두 DataFrame을 각각 저장
# 아래 df 변수명은 실제 수집 결과를 담은 변수명으로 맞추세요.
save_df(raw_station, "RAW_BIKE_STATION", conn)
save_df(raw_master_station, "RAW_BIKE_STATION_MASTER", conn)

tbCycleStationInfo: 1~1000 요청 → 1,000행 수신
tbCycleStationInfo: 1001~2000 요청 → 1,000행 수신
tbCycleStationInfo: 2001~3000 요청 → 1,000행 수신
tbCycleStationInfo: 3001~4000 요청 → 244행 수신
bikeStationMaster: 1~1000 요청 → 1,000행 수신
bikeStationMaster: 1001~2000 요청 → 1,000행 수신
bikeStationMaster: 2001~3000 요청 → 1,000행 수신
bikeStationMaster: 3001~4000 요청 → 432행 수신
대여소 정보: (3244, 13)
대여소 마스터: (3432, 5)


,STA_LOC,RENT_ID,RENT_NO,RENT_NM,RENT_ID_NM,HOLD_NUM,STA_ADD1,STA_ADD2,STA_LAT,STA_LONG,START_INDEX,END_INDEX,RNUM
0,마포구,ST-10,00108,서교동 사거리,108. 서교동 사거리,12,서울특별시 마포구 양화로 93,427,37.55274582,126.91861725,0,0,1
1,광진구,ST-100,00503,더샵스타시티 C동 앞,503. 더샵스타시티 C동 앞,15,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.53666687,127.07359314,0,0,2
2,양천구,ST-1000,00729,서부식자재마트 건너편,729. 서부식자재마트 건너편,10,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51037979,126.86679840,0,0,3
3,양천구,ST-1002,00731,서울시 도로환경관리센터,731. 서울시 도로환경관리센터,10,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.52989960,126.87654114,0,0,4
4,양천구,ST-1003,00732,신월중학교,732. 신월중학교,10,서울특별시 양천구 화곡로 59,신월동 이마트,37.53955078,126.82830048,0,0,5


,RNTLS_ID,ADDR1,ADDR2,LAT,LOT
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.510380,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.000000,0.000000
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.529900,126.876541


RAW_BIKE_STATION: 3,244행 저장 완료
RAW_BIKE_STATION_MASTER: 3,432행 저장 완료


####02-1-2. 대여이력

In [ ]:
# ============================================================
# [파트 1] 설정값 + 데이터 폴더 지정
#   방식 A: 브라우저에서 직접 업로드 (큰 파일은 느림) → /content/upload 에 저장
#   방식 B: 구글 드라이브 폴더를 그대로 읽기 (권장)
# ============================================================
USE_DRIVE  = True                          # True: 방식 B / False: 방식 A

if USE_DRIVE:
    # 방식 B: 드라이브에 올려둔 폴더
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/final_project/data/bike_raw"
    if not os.path.isdir(DATA_DIR):
        raise FileNotFoundError(f"폴더가 없습니다: {DATA_DIR}")
else:
    # 방식 A: 직접 업로드 → 업로드 폴더를 대상 폴더로 사용
    DATA_DIR = "/content/upload"
    os.makedirs(DATA_DIR, exist_ok=True)
    uploaded = files.upload(target_dir=DATA_DIR)     # 선택한 파일(ZIP/CSV/엑셀)이 DATA_DIR에 저장됨
    print("업로드 완료:", list(uploaded.keys()))
    del uploaded                                     # 업로드 원본 바이트 메모리 해제

EXCEL_EXTS = (".xlsx", ".xls")
TEXT_EXTS  = (".csv", ".tsv", ".txt")
DATA_EXTS  = EXCEL_EXTS + TEXT_EXTS
ENCODINGS  = ["utf-8-sig", "cp949", "euc-kr", "utf-8"]   # CSV 인코딩 후보(순서대로 시도)


def is_data_name(name):
    """파일명이 데이터 파일인지 판단 (폴더/맥 메타/숨김/엑셀 임시파일 제외)"""
    base = os.path.basename(name)
    return (
        not name.endswith("/")
        and not name.startswith("__MACOSX/")
        and not base.startswith((".", "~$"))
        and name.lower().endswith(DATA_EXTS)
    )


def list_sources(folder):
    """폴더를 하위까지 순회하며 처리 대상 목록 반환.
    일반 파일  → (키, 경로, None)
    ZIP 내부   → (키 'zip상대경로::내부경로', zip경로, 내부경로)
    키는 적재 이력(LOG_TABLE)의 FILE_NAME으로 사용."""
    sources = []
    for root, _, fnames in os.walk(folder):
        for fn in sorted(fnames):
            path = os.path.join(root, fn)
            rel = os.path.relpath(path, folder)
            if fn.startswith((".", "~$")):
                continue
            if fn.lower().endswith(".zip"):
                with zipfile.ZipFile(path) as z:
                    for inner in z.namelist():
                        if is_data_name(inner):
                            sources.append((f"{rel}::{inner}", path, inner))
            elif is_data_name(fn):
                sources.append((rel, path, None))
    return sources

def read_source(path, inner=None, nrows=None):
    """파일 1개를 문자열 DataFrame으로 읽기 (엑셀/CSV 자동 구분, ZIP 내부 파일 지원)"""
    name = inner or path
    lower = name.lower()

    def _open():
        if inner is None:
            return open(path, "rb")
        return zipfile.ZipFile(path).open(inner)

    if lower.endswith(EXCEL_EXTS):
        with _open() as f:
            df = pd.read_excel(f, dtype=str, nrows=nrows, keep_default_na=False)
    else:
        sep = "\t" if lower.endswith(".tsv") else ","
        df, last_err = None, None
        for enc in ENCODINGS:                       # 인코딩 후보를 차례로 시도
            try:
                with _open() as f:
                    df = pd.read_csv(f, dtype=str, nrows=nrows, sep=sep,
                                     encoding=enc, keep_default_na=False)
                break
            except UnicodeDecodeError as e:
                last_err = e
        if df is None:
            raise ValueError(f"{name}: 인코딩 판별 실패 {ENCODINGS}") from last_err

    df.columns = [str(c).strip() for c in df.columns]
    return df


In [ ]:
# ============================================================
# [파트 1-2] 대상 폴더의 파일 목록 출력 (ZIP이면 내부 파일까지 표시)
#   [적재대상] = 데이터 파일(csv/xlsx 등), [제외] = 처리하지 않는 파일
# ============================================================
def fmt_size(n):
    """바이트를 보기 좋은 단위로 변환"""
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:,.1f}{unit}"
        n /= 1024
    return f"{n:,.1f}TB"


def display_name(name):
    """ZIP 내부 한글 파일명이 깨져 보일 때(cp437 오인식) 표시용으로만 복원"""
    try:
        return name.encode("cp437").decode("cp949")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return name


def print_folder_contents(folder):
    n_files = n_zip = n_target = 0
    print(f"📁 {folder}\n")

    for root, _, fnames in sorted(os.walk(folder)):
        for fn in sorted(fnames):
            path = os.path.join(root, fn)
            rel = os.path.relpath(path, folder)
            n_files += 1

            if fn.lower().endswith(".zip"):
                n_zip += 1
                print(f"📦 {rel}  ({fmt_size(os.path.getsize(path))})")
                try:
                    with zipfile.ZipFile(path) as z:
                        infos = [i for i in z.infolist() if not i.filename.endswith("/")]
                        for info in infos:
                            is_t = is_data_name(info.filename)
                            n_target += is_t
                            tag = "[적재대상]" if is_t else "[제외]    "
                            print(f"     └ {tag} {display_name(info.filename)}  ({fmt_size(info.file_size)})")
                        print(f"     → ZIP 내부 파일 {len(infos)}개\n")
                except zipfile.BadZipFile:
                    print("     ⚠ 손상된 ZIP 파일입니다.\n")
            else:
                is_t = is_data_name(fn)
                n_target += is_t
                tag = "[적재대상]" if is_t else "[제외]    "
                print(f"📄 {tag} {rel}  ({fmt_size(os.path.getsize(path))})")

    print(f"\n폴더 내 파일 {n_files}개 (ZIP {n_zip}개) / 적재 대상 데이터 파일 총 {n_target}개")

print_folder_contents(DATA_DIR)

📁 /content/drive/MyDrive/Colab Notebooks/final_project/data/bike_raw

📦 서울특별시 공공자전거 대여이력 정보_2021.zip  (1.4GB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2109.csv  (666.6MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2110.csv  (614.6MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2111.csv  (488.0MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2112.csv  (325.6MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2102.csv  (220.9MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2101.csv  (145.1MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2103.csv  (390.6MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2104.csv  (543.7MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2105.csv  (527.0MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2106.csv  (617.7MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2107.csv  (603.0MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2108.csv  (579.1MB)
     → ZIP 내부 파일 12개

📦 서울특별시 공공자전거 대여이력 정보_2022.zip  (1.8GB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2208.csv  (648.6MB)
     └ [적재대상] 서울특별시 공공자전거 대여이력 정보_2207.csv  (734.4MB)
     └ [적재대상] 서울특별시 공공자전

In [ ]:
# ============================================================
# [파트 2] 헤더 점검 - DB 저장 전 컬럼 구성 확인용
# ============================================================
def check_headers(folder):
    sources = list_sources(folder)
    print(f"대상 파일 수: {len(sources)}개")
    print(pd.Series([os.path.splitext(k)[1].lower() for k, _, _ in sources])
          .value_counts().to_string(), "\n")

    headers, errors = {}, {}
    for key, path, inner in sources:
        try:
            headers[key] = read_source(path, inner, nrows=5).columns.tolist()  # 5행만 읽기
        except Exception as e:
            errors[key] = str(e)

    base_cols = next(iter(headers.values()))
    diff = {k: h for k, h in headers.items() if h != base_cols}
    print("기준 컬럼:", base_cols)
    print("헤더가 다른 파일:", diff if diff else "없음")
    print("읽기 실패 파일:", errors if errors else "없음")

check_headers(DATA_DIR)

대상 파일 수: 60개
.csv    60 

기준 컬럼: ['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호', '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류', '대여대여소ID', '반납대여소ID', '자전거구분']
헤더가 다른 파일: 없음
읽기 실패 파일: 없음


In [ ]:
# ============================================================
# [파트 3] 데이터 파일 → DataFrame → Oracle 저장 (기존 save_df 사용)
#   - 파일이 매우 커서(수백 MB, 파일당 수백만 행) 청크 단위로 읽고 저장
#   - 청크마다 진행 상황을 LOG_TABLE에 기록 → 중간에 끊겨도 이어서 재개
# ============================================================
TABLE_NAME = "RENTAL_HISTORY_RAW"          # 원본 데이터 테이블
LOG_TABLE  = "RENTAL_HISTORY_LOAD_LOG"     # 적재 이력 테이블 (파일별 진행 상황)
CHUNK_ROWS = 500_000                       # 한 번에 읽어서 저장할 행 수 (메모리 부족하면 줄이기)

# 원본 헤더(한글) → DB 컬럼명 매핑
COL_MAP = {
    "자전거번호":      "BIKE_NO",
    "대여일시":        "RENTAL_DATE",
    "대여 대여소번호": "RENTAL_STATION_NO",
    "대여 대여소명":   "RENTAL_STATION_NM",
    "대여거치대":      "RENTAL_HOLD",
    "반납일시":        "RETURN_DATE",
    "반납대여소번호":  "RETURN_STATION_NO",
    "반납대여소명":    "RETURN_STATION_NM",
    "반납거치대":      "RETURN_HOLD",
    "이용시간(분)":    "RENTAL_TIME",
    "이용거리(M)":     "RENTAL_DISTANCE",
    "생년":            "BIRTH",
    "성별":            "SEX",
    "이용자종류":      "USER_TYPE",
    "대여대여소ID":    "RENTAL_STATION_ID",
    "반납대여소ID":    "RETURN_STATION_ID",
    "자전거구분":      "BICYCLE_TYPE",
}

# Oracle 예약어 (컬럼명으로 쓰면 ORA-00904 발생)
RESERVED = set("""ACCESS ADD ALL ALTER AND ANY AS ASC AUDIT BETWEEN BY CHAR CHECK CLUSTER COLUMN
COMMENT COMPRESS CONNECT CREATE CURRENT DATE DECIMAL DEFAULT DELETE DESC DISTINCT DROP ELSE
EXCLUSIVE EXISTS FILE FLOAT FOR FROM GRANT GROUP HAVING IDENTIFIED IMMEDIATE IN INCREMENT INDEX
INITIAL INSERT INTEGER INTERSECT INTO IS LEVEL LIKE LOCK LONG MAXEXTENTS MINUS MLSLABEL MODE
MODIFY NOAUDIT NOCOMPRESS NOT NOWAIT NULL NUMBER OF OFFLINE ON ONLINE OPTION OR ORDER PCTFREE
PRIOR PUBLIC RAW RENAME RESOURCE REVOKE ROW ROWID ROWNUM ROWS SELECT SESSION SET SHARE SIZE
SMALLINT START SUCCESSFUL SYNONYM SYSDATE TABLE THEN TO TRIGGER UID UNION UNIQUE UPDATE USER
VALIDATE VALUES VARCHAR VARCHAR2 VIEW WHENEVER WHERE WITH""".split())


# ------------------------------------------------------------
# 3-1. 컬럼명 검사: 저장 전에 미리 걸러서 ORA-00904를 방지
# ------------------------------------------------------------
def validate_columns(cols):
    bad = [c for c in cols
           if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,29}", str(c).upper())   # 영문/숫자/_ , 30자 이내
           or str(c).upper() in RESERVED]                                  # 예약어 제외
    if bad:
        raise ValueError(f"사용할 수 없는 컬럼명(매핑 누락 또는 예약어): {bad}")


# ------------------------------------------------------------
# 3-2. 파일 읽기 헬퍼 (일반 파일 / ZIP 내부 파일 공통)
# ------------------------------------------------------------
@contextmanager
def open_source(path, inner=None):
    """일반 파일이면 그대로, ZIP이면 ZIP 내부 파일을 바이너리로 열기"""
    if inner is None:
        with open(path, "rb") as f:
            yield f
    else:
        with zipfile.ZipFile(path) as z, z.open(inner) as f:
            yield f


def detect_encoding(path, inner=None):
    """파일 앞부분 1MB로 CSV 인코딩 판별 (utf-8-sig → cp949 → euc-kr → utf-8)"""
    with open_source(path, inner) as f:
        sample = f.read(1024 * 1024)
    for enc in ENCODINGS:
        try:
            codecs.getincrementaldecoder(enc)().decode(sample, final=False)  # 잘린 끝부분은 허용
            return enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {inner or path}")


def iter_chunks(path, inner=None, chunk_rows=CHUNK_ROWS, skip_rows=0):
    """파일을 chunk_rows 행씩 잘라 DataFrame으로 반환하는 제너레이터.
    skip_rows: 이미 DB에 저장된 앞부분 행 수 (재개용, 해당 행은 건너뜀)"""
    name = (inner or path).lower()

    # 엑셀은 청크 읽기가 불가 → 통째로 읽은 뒤 잘라서 반환
    if name.endswith(EXCEL_EXTS):
        with open_source(path, inner) as f:
            df = pd.read_excel(f, dtype=str, keep_default_na=False)
        df.columns = [str(c).strip() for c in df.columns]
        for s in range(skip_rows, len(df), chunk_rows):
            yield df.iloc[s:s + chunk_rows]
        return

    # CSV/TSV/TXT: 청크 단위로 읽어 메모리 사용 최소화
    enc = detect_encoding(path, inner)
    sep = "\t" if name.endswith(".tsv") else ","
    pos = 0                                           # 지금까지 읽은 행 수
    with open_source(path, inner) as f:
        reader = pd.read_csv(f, dtype=str, sep=sep, encoding=enc,
                             keep_default_na=False, chunksize=chunk_rows)
        for chunk in reader:
            end = pos + len(chunk)
            if end <= skip_rows:                      # 이미 저장된 청크는 통째로 건너뜀
                pos = end
                continue
            if pos < skip_rows:                       # 일부만 저장된 청크는 앞부분만 잘라냄
                chunk = chunk.iloc[skip_rows - pos:]
            pos = end
            chunk.columns = [str(c).strip() for c in chunk.columns]
            yield chunk


# ------------------------------------------------------------
# 적재 이력(LOG_TABLE) 관리 - PostgreSQL 버전
#   status = 'LOADING'(진행 중/중단) | 'DONE'(완료),  row_cnt = 커밋된 행 수
# -----------------------------------------------------------
def ensure_log_table():
    with conn.cursor() as cur:
        cur.execute(f"""CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
            file_name TEXT PRIMARY KEY,
            row_cnt   BIGINT,
            status    TEXT,
            loaded_at TIMESTAMP DEFAULT now())""")
    conn.commit()


def get_log():
    """{파일키: (저장된 행 수, 상태)}"""
    with conn.cursor() as cur:
        cur.execute(f"SELECT file_name, row_cnt, status FROM {LOG_TABLE}")
        rows = {r[0]: (int(r[1] or 0), r[2]) for r in cur.fetchall()}
    conn.rollback()
    return rows



def set_log(key, row_cnt, status, commit=True):
    """파일별 진행 상황 기록 (있으면 갱신, 없으면 추가)"""
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {LOG_TABLE} (file_name, row_cnt, status) VALUES (%s, %s, %s)
            ON CONFLICT (file_name) DO UPDATE
            SET row_cnt = EXCLUDED.row_cnt, status = EXCLUDED.status, loaded_at = now()""",
            [key, row_cnt, status])
    if commit:
        conn.commit()


# ------------------------------------------------------------
# 파일 1개 적재: 청크 읽기 → 컬럼명 변환 → save_df → 로그 기록
#   데이터와 로그를 한 트랜잭션으로 커밋 → 중간에 끊겨도 중복/누락 없음
# ------------------------------------------------------------
def load_one_file(key, path, inner, done_rows=0):
    loaded = done_rows
    first = True
    try:
        for chunk in iter_chunks(path, inner, CHUNK_ROWS, skip_rows=done_rows):
            ensure_conn()                                   # 끊겼으면 재접속
            chunk = chunk.rename(columns=COL_MAP)           # 한글 → 영문 컬럼명
            if first:
                validate_columns(chunk.columns)             # 매핑 누락/예약어 검사
                first = False

            save_df(chunk, TABLE_NAME, conn, mode="INSERT", commit=False)  # 커밋 보류
            loaded += len(chunk)
            set_log(key, loaded, "LOADING", commit=False)
            conn.commit()                                   # 데이터 + 로그 동시 커밋
        ensure_conn()
        set_log(key, loaded, "DONE")
        return loaded
    except BaseException:
        try:
            conn.rollback()                                 # 미커밋 데이터 정리
        except Exception:
            pass
        raise

# ------------------------------------------------------------
# 폴더 전체 반복 적재 (파일 목록 → 반복문 → 파일별 적재)
# ------------------------------------------------------------
def load_folder_to_db(folder, max_files=None, stop_on_error=True):
    validate_columns(COL_MAP.values())
    ensure_conn()
    ensure_log_table()

    sources = list_sources(folder)
    log = get_log()
    todo = []
    for key, path, inner in sorted(sources):                # 파일키 순서 = 연도/월 순서
        rows, status = log.get(key, (0, None))
        if status == "DONE":                                # 완료된 파일은 건너뜀
            continue
        todo.append((key, path, inner, rows))               # 중단된 파일은 이어서
    if max_files:
        todo = todo[:max_files]

    n_done = sum(1 for v in log.values() if v[1] == "DONE")
    print(f"전체 {len(sources)}개 / 완료 {n_done}개 / 이번 처리 {len(todo)}개")

    failed = {}
    for i, (key, path, inner, done_rows) in enumerate(todo, 1):          # ← 파일별 반복문
        resume = f" (이어서: {done_rows:,}행 이후)" if done_rows else ""
        print(f"\n[{i}/{len(todo)}] {key}{resume}")
        t0 = time.time()
        try:
            n = load_one_file(key, path, inner, done_rows)
            print(f"   ✔ 완료: 파일 누적 {n:,}행 / {time.time()-t0:,.0f}초")
        except Exception as e:
            failed[key] = str(e)
            print(f"   ⚠ 실패: {e}")
            if stop_on_error:
                print("   → 중단합니다. 원인 해결 후 다시 실행하면 이어서 진행합니다.")
                break

    print(f"\n처리 종료: 실패 {len(failed)}개", failed if failed else "")
    return failed

In [ ]:
# 전체 파일 저장
failed = load_folder_to_db(DATA_DIR, max_files=None)

conn.rollback()      # 방금 오류로 중단된 트랜잭션 정리 (필수)

with conn.cursor() as cur:
    # 1) 적재 이력: 완료 파일 수와 로그상 총 행 수
    cur.execute(f"SELECT COUNT(*) FILTER (WHERE status = 'DONE'), COUNT(*), SUM(row_cnt) FROM {LOG_TABLE}")
    done, total, log_rows = cur.fetchone()
    print(f"완료 파일 {done}/{total}개 / 로그상 총 행 수: {int(log_rows):,}")

    # 2) 테이블 용량
    cur.execute("SELECT pg_size_pretty(pg_total_relation_size('rental_history_raw'))")
    print("테이블 용량:", cur.fetchone()[0])

    # 3) 실제 행 수 (수억 행이라 수십 초~수 분 걸릴 수 있음)
    cur.execute("SELECT COUNT(*) FROM rental_history_raw")
    real_rows = cur.fetchone()[0]
    print(f"실제 행 수: {real_rows:,}  →  로그와 {'일치 ✔' if real_rows == int(log_rows) else '불일치 ⚠'}")

    # 4) 컬럼과 샘플 확인
    cur.execute("SELECT * FROM rental_history_raw LIMIT 3")
    print("컬럼:", [d[0] for d in cur.description])
    for r in cur.fetchall():
        print(" ", r)
conn.rollback()

[재접속 완료]
전체 60개 / 완료 60개 / 이번 처리 0개

처리 종료: 실패 0개 
완료 파일 60/60개 / 로그상 총 행 수: 199,133,465
테이블 용량: 40 GB
실제 행 수: 199,133,465  →  로그와 일치 ✔
컬럼: ['bike_no', 'rental_date', 'rental_station_no', 'rental_station_nm', 'rental_hold', 'return_date', 'return_station_no', 'return_station_nm', 'return_hold', 'rental_time', 'rental_distance', 'birth', 'sex', 'user_type', 'rental_station_id', 'return_station_id', 'bicycle_type']
  ('SPB-54051', '2021-01-01 00:02:50', '01064', '중앙보훈병원역 1번출구', '0', '2021-01-01 00:04:51', '01048', '둔촌어린이공원', '0', '2', '0.00', '1978', 'M', '내국인', 'ST-1763', 'ST-1370', '일반자전거')
  ('SPB-31575', '2021-01-01 00:00:33', '01977', '천왕역 1번 출입구 앞', '0', '2021-01-01 00:06:03', '01952', '천왕연지타운2단지 앞', '0', '5', '810.91', '1976', '\\N', '내국인', 'ST-1239', 'ST-875', '일반자전거')
  ('SPB-44420', '2021-01-01 00:01:46', '03011', '경의선(노고산동)', '0', '2021-01-01 00:08:04', '00120', '신수동 사거리', '0', '6', '920.27', '1980', '\\N', '내국인', 'ST-2169', 'ST-22', '일반자전거')


####02-1-3. 고장신고

#고장신고는중간 멈춤 다시할것

In [ ]:
# [고장신고] 날짜별 API 수집 → PostgreSQL 저장
TABLE_NAME = "BIKE_FAULT_REPORT"
MAX_RETRIES = 3

# 같은 런타임에서 다시 실행하면 마지막 미완료 날짜부터 재개
# 처음 실행할 때는 START_DATE부터 시작
if "FAILURE_NEXT_DATE" not in globals():
    FAILURE_NEXT_DATE = START_DATE

print(f"수집 범위: {FAILURE_NEXT_DATE} ~ {END_DATE}")

saved_rows = 0

for date in pd.date_range(FAILURE_NEXT_DATE, END_DATE, freq="D"):
    date_text = date.strftime("%Y-%m-%d")
    date_parameter = date.strftime("%Y%m%d")

    # 실패하면 이 날짜부터 다시 실행하도록 기록
    FAILURE_NEXT_DATE = date_text
    df_day = None

    print(f"\n[고장신고] 조회 날짜: {date_text}")

    # 1. 하루치 수집: 통신/JSON 응답 실패 시 최대 3회 시도
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df_day = fetch_all_rows(
                service="tbCycleFailureReport",
                date_parameter=date_parameter
            )
            break

        except RuntimeError as error:
            # 기존 fetch_all_rows()에서 발생시키는 요청 실패만 재시도
            # 인증키 오류, API 오류 응답 등은 그대로 중단
            if "행 요청 실패" not in str(error):
                raise

            print(f"[요청 실패] {attempt}/{MAX_RETRIES}회")

            if attempt == MAX_RETRIES:
                print(f"[중단] 재개할 날짜: {FAILURE_NEXT_DATE}")
                raise

            wait_seconds = attempt * 3
            print(f"{wait_seconds}초 후 해당 날짜를 다시 조회합니다.")
            time.sleep(wait_seconds)

    # 2. 하루치 수집이 끝난 뒤에만 DB에 저장
    print(f"수신 건수: {len(df_day):,}행")

    if not df_day.empty:
        try:
            save_df(
                df=df_day,
                table_name=TABLE_NAME,
                conn=conn,
                mode="INSERT"
            )
            saved_rows += len(df_day)

        except Exception:
            print(f"[DB 저장 실패] 날짜: {date_text}")
            print("API 수집은 완료됐으며, df_day에 데이터가 남아 있습니다.")
            raise

    else:
        print("[조회 결과 없음] 저장하지 않습니다.")

    # 3. 저장 성공 또는 조회 결과 없음일 때 다음 날짜로 이동
    FAILURE_NEXT_DATE = (
        date + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")

    print(f"[처리 완료] 다음 조회 날짜: {FAILURE_NEXT_DATE}")
    time.sleep(0.3)

print(f"\n이번 실행에서 DB에 저장한 데이터: {saved_rows:,}행")
print(f"다음 조회 날짜: {FAILURE_NEXT_DATE}")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
tbCycleFailureReport: 11001~12000 요청 → 1,000행 수신
tbCycleFailureReport: 12001~13000 요청 → 1,000행 수신
tbCycleFailureReport: 13001~14000 요청 → 1,000행 수신
tbCycleFailureReport: 14001~15000 요청 → 1,000행 수신
tbCycleFailureReport: 15001~16000 요청 → 1,000행 수신
tbCycleFailureReport: 16001~17000 요청 → 1,000행 수신
tbCycleFailureReport: 17001~18000 요청 → 1,000행 수신
tbCycleFailureReport: 18001~19000 요청 → 1,000행 수신
tbCycleFailureReport: 19001~20000 요청 → 1,000행 수신
tbCycleFailureReport: 20001~21000 요청 → 805행 수신
수신 건수: 20,805행
BIKE_FAULT_REPORT: 20,805행 저장 완료
[처리 완료] 다음 조회 날짜: 2024-09-19

[고장신고] 조회 날짜: 2024-09-19
tbCycleFailureReport: 1~1000 요청 → 1,000행 수신
tbCycleFailureReport: 1001~2000 요청 → 1,000행 수신
tbCycleFailureReport: 2001~3000 요청 → 1,000행 수신
tbCycleFailureReport: 3001~4000 요청 → 1,000행 수신
tbCycleFailureReport: 4001~5000 요청 → 1,000행 수신
tbCycleFailureReport: 5001~6000 요청 → 1,000행 수신
tbCycleFailureReport: 6001~7000 요청 → 1,000행 수신
tbCycleFailureReport: 7001~8000 요청 → 1,000행 수신


##02-2. 지하철 승하차

* 출퇴근시간을 정의
* 출퇴근시간은 따릉이의 이용빈도에 영향을 줄까?
  > 역 근처의 따릉이를 확인. 이용빈도를 같이 확인+ 지도필요.





* API로 받은 원본 그대로 PostgreSQL 테이블에 만들고 저장 (파일로 저장하지 않음)
* 아래 셀은 앞의 셀이 먼저 실행되어 있어야 함 (conn, ensure_conn, save_df, fetch_all_rows, check_api, SEOUL_API_KEY, START_DATE, END_DATE)
* 컬럼은 API 응답 컬럼명을 소문자로 바꿔 전부 TEXT로 저장

In [ ]:
RAW_SOURCES = {
    "card_time_monthly": {                     # 서울시 월별·시간대별 승하차 (OA-12252)
        "service": "CardSubwayTime",
        "table":   "RAW_SUBWAY_CARD_TIME",
        "freq":    "M",
    },
    "daily_total": {                           # 서울시 지하철호선별 역별 승하차 인원, 일별 (OA-12914)
        "service": "CardSubwayStatsNew",       # ※ 미확인 후보: 확인 셀에서 검증할 것
        "table":   "RAW_SUBWAY_DAILY_TOTAL",
        "freq":    "D",
    },
    "metro_daily_hour": {                      # 서울교통공사 일별·시간대별 승하차 (OA-12921)
        "service": "subwSttn",                         # ← OpenAPI 서비스명이 있을 때만 입력
        "table":   "RAW_SUBWAY_METRO_DAILY_HOUR",
        "freq":    "D",
    },
}
LOAD_LOG     = "SUBWAY_RAW_LOAD_LOG"           # 데이터셋 + 기간별 적재 이력 테이블
MAX_RETRIES  = 3                               # 통신 실패 시 재시도 횟수
PERIOD_FORMAT = {"D": "%Y%m%d", "M": "%Y%m", "": None}


In [ ]:
def check_sources(names=None):
    """설정 확인: 서비스명이 비어 있거나 freq가 잘못되면 호출 전에 중단"""
    for name in (names or RAW_SOURCES):
        cfg = RAW_SOURCES[name]
        if not cfg["service"].strip():
            raise ValueError(
                f"[{name}] service가 비어 있습니다. 데이터셋 페이지 OpenAPI 탭의 "
                f"샘플url에서 서비스명을 복사해 RAW_SOURCES에 넣으세요.")
        if cfg["freq"] not in PERIOD_FORMAT:
            raise ValueError(f"[{name}] freq는 'D', 'M', '' 중 하나여야 합니다: {cfg['freq']!r}")


def check_source(name, date_text="2024-01-02"):
    """api 정상접속 확인 : 첫 5행만 조회해서 컬럼 / 기간 / 단위 확인"""
    check_sources([name])
    cfg = RAW_SOURCES[name]
    fmt = PERIOD_FORMAT[cfg["freq"]]
    param = pd.Timestamp(date_text).strftime(fmt) if fmt else ""
    return check_api(service=cfg["service"], date_parameter=param)

# 테이블 존재 확인
def table_exists(table_name):
    with conn.cursor() as cur:
        cur.execute("SELECT to_regclass(%s::text)", [table_name.lower()])
        exists = cur.fetchone()[0] is not None
    conn.rollback()                                 # 확인용 트랜잭션 정리
    return exists

In [ ]:
# 기간별 수집 후 저장 함수
def ensure_load_log():
    ensure_conn()
    with conn.cursor() as cur:
        cur.execute(f"""CREATE TABLE IF NOT EXISTS {LOAD_LOG} (
            dataset   TEXT NOT NULL,
            period    TEXT NOT NULL,
            row_cnt   BIGINT,
            status    TEXT,
            loaded_at TIMESTAMP DEFAULT now(),
            PRIMARY KEY (dataset, period))""")
    conn.commit()


def get_load_log(dataset):
    """{기간: 상태}"""
    with conn.cursor() as cur:
        cur.execute(f"SELECT period, status FROM {LOAD_LOG} WHERE dataset = %s", [dataset])
        log = {r[0]: r[1] for r in cur.fetchall()}
    conn.rollback()
    return log


def set_load_log(dataset, period, row_cnt, status, commit=True):
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {LOAD_LOG} (dataset, period, row_cnt, status) VALUES (%s, %s, %s, %s)
            ON CONFLICT (dataset, period) DO UPDATE
            SET row_cnt = EXCLUDED.row_cnt, status = EXCLUDED.status, loaded_at = now()""",
            [dataset, period, row_cnt, status])
    if commit:
        conn.commit()


def fetch_period(service, param):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fetch_all_rows(service, param)
        except RuntimeError as error:
            if "행 요청 실패" not in str(error):
                raise
            print(f"[요청 실패] {attempt}/{MAX_RETRIES}회")
            if attempt == MAX_RETRIES:
                raise
            wait_seconds = attempt * 3
            print(f"{wait_seconds}초 후 {param or '전체'}를 다시 조회합니다.")
            time.sleep(wait_seconds)


def load_period(name, period, param):
    cfg = RAW_SOURCES[name]
    df_period = fetch_period(cfg["service"], param)

    ensure_conn()                                       # 수집하는 동안 끊겼으면 재접속
    try:
        if df_period.empty:
            # 빈 응답을 '승하차 0명'으로 확정하지 않고 기록만 남김
            set_load_log(name, period, 0, "EMPTY", commit=False)
        else:
            save_df(df_period, cfg["table"], conn, mode="INSERT", commit=False)
            set_load_log(name, period, len(df_period), "DONE", commit=False)
        conn.commit()                                   # 데이터 + 이력 동시 커밋
    except Exception:
        conn.rollback()
        raise
    return len(df_period)

In [ ]:
# 기간 전체 반복 적재 함수
def load_source(name, max_periods=None, retry_empty=False, stop_on_error=True):
    check_sources([name])
    cfg = RAW_SOURCES[name]
    fmt = PERIOD_FORMAT[cfg["freq"]]
    ensure_conn()
    ensure_load_log()

    if fmt is None:                                     # 조건값 없는 데이터셋: 1회 전체 조회
        periods = [("ALL", "")]
    else:
        periods = [(p.strftime(fmt), p.strftime(fmt))
                   for p in pd.period_range(START_DATE, END_DATE, freq=cfg["freq"])]

    log = get_load_log(name)
    skip = {"DONE"} if retry_empty else {"DONE", "EMPTY"}
    todo = [(p, q) for p, q in periods if log.get(p) not in skip]
    if max_periods:
        todo = todo[:max_periods]

    n_done = sum(1 for p, _ in periods if log.get(p) == "DONE")
    print(f"[{name}] 전체 {len(periods)}개 / 완료 {n_done}개 / 이번 처리 {len(todo)}개")

    failed = {}
    for i, (period, param) in enumerate(todo, 1):       # ← 기간별 반복문
        print(f"\n[{name} {i}/{len(todo)}] {period}")
        t0 = time.time()
        try:
            n = load_period(name, period, param)
            print(f"   ✔ {n:,}행 / {time.time()-t0:,.0f}초")
        except Exception as e:
            failed[period] = str(e)
            print(f"   ⚠ {period} 실패: {e}")
            if stop_on_error:
                print("   → 중단합니다. 원인 해결 후 다시 실행하면 이 기간부터 이어서 진행합니다.")
                break
        time.sleep(0.2)

    print(f"\n[{name}] 처리 종료: 실패 {len(failed)}개", failed if failed else "")
    return failed

###02-2-1. 지하철 역 정보

In [ ]:
# api 정상접속 확인
df_subway_master_check = check_api(service="subwayStationMaster")


서비스: subwayStationMaster
응답 전체 건수: 784
이번에 받은 행 수: 5
원본 컬럼: ['BLDN_ID', 'BLDN_NM', 'ROUTE', 'LAT', 'LOT']


,BLDN_ID,BLDN_NM,ROUTE,LAT,LOT
0,0150,서울역,1호선,37.556228,126.972135
1,0151,시청,1호선,37.565715,126.977088
2,0152,종각,1호선,37.570161,126.982923
3,0153,종로3가,1호선,37.570406,126.991847
4,0154,종로5가,1호선,37.570926,127.001849


In [ ]:
#역 정보 DB 저장
MASTER_TABLE = "RAW_SUBWAY_STATION_MASTER"

if table_exists(MASTER_TABLE):
    print(f"{MASTER_TABLE}: 이미 있는 테이블이라 저장하지 않습니다.")
    print("다시 넣으려면 DROP TABLE 후 이 셀을 다시 실행하세요.")
else:
    raw_subway_master = fetch_all_rows("subwayStationMaster")   # 1,000행씩 끝까지 수집

    print("역사마스터:", raw_subway_master.shape)
    print("원본 컬럼:", raw_subway_master.columns.tolist())
    display(raw_subway_master.head())

    if raw_subway_master.empty:
        raise RuntimeError("수집된 역사마스터 데이터가 없어 저장하지 않습니다.")

    save_df(raw_subway_master, MASTER_TABLE, conn)

subwayStationMaster: 1~1000 요청 → 784행 수신
역사마스터: (784, 5)
원본 컬럼: ['BLDN_ID', 'BLDN_NM', 'ROUTE', 'LAT', 'LOT']


,BLDN_ID,BLDN_NM,ROUTE,LAT,LOT
0,0150,서울역,1호선,37.556228,126.972135
1,0151,시청,1호선,37.565715,126.977088
2,0152,종각,1호선,37.570161,126.982923
3,0153,종로3가,1호선,37.570406,126.991847
4,0154,종로5가,1호선,37.570926,127.001849


RAW_SUBWAY_STATION_MASTER: 784행 저장 완료


###02-2-2. 월별승하차 정보

* 교통카드 정산시스템 시간대별 승하차 (CardSubwayTime, OA-12252) > 출처: 서울 열린데이터광장

In [ ]:
# api 정상접속 확인
df_subway_time_check = check_source("card_time_monthly", "2021-01-01")


서비스: CardSubwayTime
응답 전체 건수: 604
이번에 받은 행 수: 5
원본 컬럼: ['USE_MM', 'SBWY_ROUT_LN_NM', 'STTN', 'HR_4_GET_ON_NOPE', 'HR_4_GET_OFF_NOPE', 'HR_5_GET_ON_NOPE', 'HR_5_GET_OFF_NOPE', 'HR_6_GET_ON_NOPE', 'HR_6_GET_OFF_NOPE', 'HR_7_GET_ON_NOPE', 'HR_7_GET_OFF_NOPE', 'HR_8_GET_ON_NOPE', 'HR_8_GET_OFF_NOPE', 'HR_9_GET_ON_NOPE', 'HR_9_GET_OFF_NOPE', 'HR_10_GET_ON_NOPE', 'HR_10_GET_OFF_NOPE', 'HR_11_GET_ON_NOPE', 'HR_11_GET_OFF_NOPE', 'HR_12_GET_ON_NOPE', 'HR_12_GET_OFF_NOPE', 'HR_13_GET_ON_NOPE', 'HR_13_GET_OFF_NOPE', 'HR_14_GET_ON_NOPE', 'HR_14_GET_OFF_NOPE', 'HR_15_GET_ON_NOPE', 'HR_15_GET_OFF_NOPE', 'HR_16_GET_ON_NOPE', 'HR_16_GET_OFF_NOPE', 'HR_17_GET_ON_NOPE', 'HR_17_GET_OFF_NOPE', 'HR_18_GET_ON_NOPE', 'HR_18_GET_OFF_NOPE', 'HR_19_GET_ON_NOPE', 'HR_19_GET_OFF_NOPE', 'HR_20_GET_ON_NOPE', 'HR_20_GET_OFF_NOPE', 'HR_21_GET_ON_NOPE', 'HR_21_GET_OFF_NOPE', 'HR_22_GET_ON_NOPE', 'HR_22_GET_OFF_NOPE', 'HR_23_GET_ON_NOPE', 'HR_23_GET_OFF_NOPE', 'HR_0_GET_ON_NOPE', 'HR_0_GET_OFF_NOPE', 'HR_1_GET_ON_NOPE

,USE_MM,SBWY_ROUT_LN_NM,STTN,HR_4_GET_ON_NOPE,HR_4_GET_OFF_NOPE,HR_5_GET_ON_NOPE,HR_5_GET_OFF_NOPE,HR_6_GET_ON_NOPE,HR_6_GET_OFF_NOPE,HR_7_GET_ON_NOPE,...,HR_23_GET_OFF_NOPE,HR_0_GET_ON_NOPE,HR_0_GET_OFF_NOPE,HR_1_GET_ON_NOPE,HR_1_GET_OFF_NOPE,HR_2_GET_ON_NOPE,HR_2_GET_OFF_NOPE,HR_3_GET_ON_NOPE,HR_3_GET_OFF_NOPE,JOB_YMD
0,202101,1호선,서울역,445.0,8.0,6832.0,4781.0,8835.0,24937.0,27116.0,...,3751.0,32.0,398.0,0.0,2.0,0.0,0.0,0.0,0.0,20210203
1,202101,1호선,동묘앞,78.0,2.0,2391.0,738.0,2761.0,3527.0,4457.0,...,1502.0,1.0,704.0,0.0,8.0,0.0,0.0,0.0,0.0,20210203
2,202101,1호선,시청,18.0,1.0,1489.0,2990.0,2235.0,14570.0,4971.0,...,629.0,4.0,48.0,0.0,1.0,0.0,0.0,0.0,0.0,20210203
3,202101,1호선,종각,113.0,1.0,1852.0,3464.0,2578.0,18288.0,4413.0,...,785.0,10.0,100.0,1.0,1.0,0.0,0.0,0.0,0.0,20210203
4,202101,1호선,종로3가,109.0,4.0,2187.0,2427.0,2374.0,8473.0,3297.0,...,1104.0,7.0,194.0,0.0,3.0,0.0,0.0,0.0,0.0,20210203


In [ ]:
# 1개월만 시험 저장
failed = load_source("card_time_monthly", max_periods=1)

[card_time_monthly] 전체 69개 / 완료 0개 / 이번 처리 1개

[card_time_monthly 1/1] 202101
CardSubwayTime: 1~1000 요청 → 604행 수신
RAW_SUBWAY_CARD_TIME: 604행 저장 완료
   ✔ 604행 / 2초

[card_time_monthly] 처리 종료: 실패 0개 


In [ ]:
# 이전 버전 이력 이전
OLD_LOG = "SUBWAY_CARD_TIME_LOAD_LOG"
if table_exists(OLD_LOG):
    ensure_load_log()
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {LOAD_LOG} (dataset, period, row_cnt, status, loaded_at)
            SELECT 'card_time_monthly', use_month, row_cnt, status, loaded_at FROM {OLD_LOG}
            ON CONFLICT (dataset, period) DO NOTHING""")
        print("이전 이력 이전:", cur.rowcount, "건")
    conn.commit()
else:
    print("이전 버전 이력 테이블이 없어 건너뜁니다.")
    #전체 기간 저장
    failed = load_source("card_time_monthly")

이전 버전 이력 테이블이 없어 건너뜁니다.
[card_time_monthly] 전체 69개 / 완료 1개 / 이번 처리 68개

[card_time_monthly 1/68] 202102
CardSubwayTime: 1~1000 요청 → 605행 수신
RAW_SUBWAY_CARD_TIME: 605행 저장 완료
   ✔ 605행 / 2초

[card_time_monthly 2/68] 202103
CardSubwayTime: 1~1000 요청 → 606행 수신
RAW_SUBWAY_CARD_TIME: 606행 저장 완료
   ✔ 606행 / 3초

[card_time_monthly 3/68] 202104
CardSubwayTime: 1~1000 요청 → 608행 수신
RAW_SUBWAY_CARD_TIME: 608행 저장 완료
   ✔ 608행 / 3초

[card_time_monthly 4/68] 202105
CardSubwayTime: 1~1000 요청 → 608행 수신
RAW_SUBWAY_CARD_TIME: 608행 저장 완료
   ✔ 608행 / 2초

[card_time_monthly 5/68] 202106
CardSubwayTime: 1~1000 요청 → 608행 수신
RAW_SUBWAY_CARD_TIME: 608행 저장 완료
   ✔ 608행 / 2초

[card_time_monthly 6/68] 202107
CardSubwayTime: 1~1000 요청 → 607행 수신
RAW_SUBWAY_CARD_TIME: 607행 저장 완료
   ✔ 607행 / 2초

[card_time_monthly 7/68] 202108
CardSubwayTime: 1~1000 요청 → 607행 수신
RAW_SUBWAY_CARD_TIME: 607행 저장 완료
   ✔ 607행 / 3초

[card_time_monthly 8/68] 202109
CardSubwayTime: 1~1000 요청 → 607행 수신
RAW_SUBWAY_CARD_TIME: 607행 저장 완료
   ✔ 607

###02-2-3. 일별 승하차 정보


* 월별 자료만으로는 일별 분석이 안 되어 일별 자료를 원본 그대로 추가 저장
* 1. 서울시 지하철호선별 역별 승하차 인원, 일별 (OA-12914) > 추정값 보정용 실제 일별 총량
* 2. 서울교통공사 일별·시간대별 승하차 (OA-12921) > 날짜·요일에 따른 시간대별 이용 패턴 파악용
   * 파일(CSV)로만 제공되는 경우 API 대신 파일을 읽어 같은 방식으로 DB에 저장해야 함


In [ ]:
# 서비스명이 틀렸다면 여기서 ERROR 가 나므로 전체 수집 전에 알 수 있음
df_total_check = check_source("daily_total")

[CardSubwayStatsNew] 조회된 자료 없음: 20240102


In [ ]:
# 3개 기간만 시험 저장
failed = load_source("daily_total", max_periods=3)

[daily_total] 전체 2099개 / 완료 0개 / 이번 처리 3개

[daily_total 1/3] 20210101
   ✔ 0행 / 1초

[daily_total 2/3] 20210102
   ✔ 0행 / 1초

[daily_total 3/3] 20210103
   ✔ 0행 / 1초

[daily_total] 처리 종료: 실패 0개 


In [ ]:
# 전체 기간 저장 (끊기면 다시 실행)
failed = load_source("daily_total")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
[daily_total 574/2096] 20220731
   ✔ 0행 / 1초

[daily_total 575/2096] 20220801
   ✔ 0행 / 1초

[daily_total 576/2096] 20220802
   ✔ 0행 / 1초

[daily_total 577/2096] 20220803
   ✔ 0행 / 1초

[daily_total 578/2096] 20220804
   ✔ 0행 / 1초

[daily_total 579/2096] 20220805
   ✔ 0행 / 1초

[daily_total 580/2096] 20220806
   ✔ 0행 / 1초

[daily_total 581/2096] 20220807
   ✔ 0행 / 1초

[daily_total 582/2096] 20220808
   ✔ 0행 / 1초

[daily_total 583/2096] 20220809
   ✔ 0행 / 1초

[daily_total 584/2096] 20220810
   ✔ 0행 / 1초

[daily_total 585/2096] 20220811
   ✔ 0행 / 1초

[daily_total 586/2096] 20220812
   ✔ 0행 / 1초

[daily_total 587/2096] 20220813
   ✔ 0행 / 1초

[daily_total 588/2096] 20220814
   ✔ 0행 / 1초

[daily_total 589/2096] 20220815
   ✔ 0행 / 1초

[daily_total 590/2096] 20220816
   ✔ 0행 / 1초

[daily_total 591/2096] 20220817
   ✔ 0행 / 1초

[daily_total 592/2096] 20220818
   ✔ 0행 / 1초

[daily_total 593/2096] 20220819
   ✔ 0행 / 1초

[daily_total 594/2096] 20220820
   ✔ 0행 / 1초

In [ ]:
# 서울교통공사 일별·시간대별
df_metro_check = check_source("metro_daily_hour")
failed = load_source("metro_daily_hour", max_periods=3)


서비스: subwSttn
응답 전체 건수: 398680
이번에 받은 행 수: 5
원본 컬럼: ['SN', 'MVMN_YMD', 'LINE', 'STTN_NO', 'STTN', 'GTNF_SE', 'HR06_BFR', 'HR06', 'HR07', 'HR08', 'HR09', 'HR10', 'HR11', 'HR12', 'HR13', 'HR14', 'HR15', 'HR16', 'HR17', 'HR18', 'HR19', 'HR20', 'HR21', 'HR22', 'HR23', 'HR24']


,SN,MVMN_YMD,LINE,STTN_NO,STTN,GTNF_SE,HR06_BFR,HR06,HR07,HR08,...,HR15,HR16,HR17,HR18,HR19,HR20,HR21,HR22,HR23,HR24
0,1,2024-01-01,1호선,150,서울역,승차,383.0,257.0,308.0,975.0,...,2716.0,2882.0,2871.0,2685.0,2922.0,2031.0,2279.0,1729.0,868.0,43.0
1,2,2024-01-01,1호선,150,서울역,하차,249.0,867.0,834.0,1201.0,...,2615.0,2501.0,2829.0,2095.0,1833.0,1465.0,1031.0,585.0,298.0,82.0
2,3,2024-01-01,1호선,151,시청,승차,188.0,92.0,167.0,245.0,...,935.0,1003.0,978.0,1116.0,951.0,909.0,723.0,462.0,176.0,3.0
3,4,2024-01-01,1호선,151,시청,하차,103.0,276.0,292.0,451.0,...,834.0,697.0,670.0,564.0,351.0,302.0,272.0,120.0,89.0,38.0
4,5,2024-01-01,1호선,152,종각,승차,970.0,374.0,193.0,205.0,...,1383.0,1578.0,1435.0,1422.0,1287.0,1318.0,1106.0,775.0,274.0,8.0


[재접속 완료]
[metro_daily_hour] 전체 2099개 / 완료 0개 / 이번 처리 3개

[metro_daily_hour 1/3] 20210101
subwSttn: 1~1000 요청 → 1,000행 수신
subwSttn: 1001~2000 요청 → 1,000행 수신
subwSttn: 2001~3000 요청 → 1,000행 수신
subwSttn: 3001~4000 요청 → 1,000행 수신
subwSttn: 4001~5000 요청 → 1,000행 수신
subwSttn: 5001~6000 요청 → 1,000행 수신
subwSttn: 6001~7000 요청 → 1,000행 수신
subwSttn: 7001~8000 요청 → 1,000행 수신
subwSttn: 8001~9000 요청 → 1,000행 수신
subwSttn: 9001~10000 요청 → 1,000행 수신
subwSttn: 10001~11000 요청 → 1,000행 수신
subwSttn: 11001~12000 요청 → 1,000행 수신
subwSttn: 12001~13000 요청 → 1,000행 수신
subwSttn: 13001~14000 요청 → 1,000행 수신
subwSttn: 14001~15000 요청 → 1,000행 수신
subwSttn: 15001~16000 요청 → 1,000행 수신
subwSttn: 16001~17000 요청 → 1,000행 수신
subwSttn: 17001~18000 요청 → 1,000행 수신
subwSttn: 18001~19000 요청 → 1,000행 수신
subwSttn: 19001~20000 요청 → 1,000행 수신
subwSttn: 20001~21000 요청 → 1,000행 수신
subwSttn: 21001~22000 요청 → 1,000행 수신
subwSttn: 22001~23000 요청 → 1,000행 수신
subwSttn: 23001~24000 요청 → 1,000행 수신
subwSttn: 24001~25000 요청 → 1,000행 수신
subwSttn

### 저장결과 확인

In [ ]:
conn.rollback()      # 오류로 중단된 트랜잭션이 남아 있으면 정리 (필수)

for name, cfg in RAW_SOURCES.items():
    table = cfg["table"].lower()
    print(f"\n===== {name} ({table}) =====")
    if not table_exists(cfg["table"]):
        print("테이블이 아직 없습니다.")
        continue
    with conn.cursor() as cur:
        cur.execute(f"SELECT status, COUNT(*), SUM(row_cnt) FROM {LOAD_LOG} "
                    "WHERE dataset = %s GROUP BY status ORDER BY status", [name])
        for status, n_periods, n_rows in cur.fetchall():
            print(f"{status}: {n_periods}개 기간 / {int(n_rows or 0):,}행")

        cur.execute(f"SELECT COALESCE(SUM(row_cnt), 0) FROM {LOAD_LOG} "
                    "WHERE dataset = %s AND status = 'DONE'", [name])
        log_rows = int(cur.fetchone()[0])
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        real_rows = cur.fetchone()[0]
        print(f"실제 행 수: {real_rows:,}  →  로그와 {'일치 ✔' if real_rows == log_rows else '불일치 ⚠'}")

        cur.execute(f"SELECT * FROM {table} LIMIT 3")
        print("컬럼:", [d[0] for d in cur.description])
        for r in cur.fetchall():
            print(" ", r)
    conn.rollback()


===== card_time_monthly (raw_subway_card_time) =====
DONE: 69개 기간 / 43,760행
실제 행 수: 43,760  →  로그와 일치 ✔
컬럼: ['use_mm', 'sbwy_rout_ln_nm', 'sttn', 'hr_4_get_on_nope', 'hr_4_get_off_nope', 'hr_5_get_on_nope', 'hr_5_get_off_nope', 'hr_6_get_on_nope', 'hr_6_get_off_nope', 'hr_7_get_on_nope', 'hr_7_get_off_nope', 'hr_8_get_on_nope', 'hr_8_get_off_nope', 'hr_9_get_on_nope', 'hr_9_get_off_nope', 'hr_10_get_on_nope', 'hr_10_get_off_nope', 'hr_11_get_on_nope', 'hr_11_get_off_nope', 'hr_12_get_on_nope', 'hr_12_get_off_nope', 'hr_13_get_on_nope', 'hr_13_get_off_nope', 'hr_14_get_on_nope', 'hr_14_get_off_nope', 'hr_15_get_on_nope', 'hr_15_get_off_nope', 'hr_16_get_on_nope', 'hr_16_get_off_nope', 'hr_17_get_on_nope', 'hr_17_get_off_nope', 'hr_18_get_on_nope', 'hr_18_get_off_nope', 'hr_19_get_on_nope', 'hr_19_get_off_nope', 'hr_20_get_on_nope', 'hr_20_get_off_nope', 'hr_21_get_on_nope', 'hr_21_get_off_nope', 'hr_22_get_on_nope', 'hr_22_get_off_nope', 'hr_23_get_on_nope', 'hr_23_get_off_nope', 'hr_0

# 아래는 안함!

##02-3. 기상예보

* 날씨가 따릉이의 이용빈도에 영향을 줄까?

##02-4. 도로상황

* 도로교통상황이 따릉이의 이용빈도에 영향을 줄까?
* 따릉이를 옮길 때 최적의 도로 시간대는?